# Hushwheel Fixture RAG Lab

This notebook turns the whimsical hushwheel corpus into a repeatable retrieval lab using the same tested helpers that back the repository tests.

## 1. Load The Shared Fixture Scaffold

### 1.1 Keep Fixture Logic In Tested Python Modules

The notebook should orchestrate the experiment, not duplicate benchmark, validation, or corpus-summary logic inline.

In [ ]:
from pathlib import Path

from repo_rag_lab.notebook_scaffolding import build_hushwheel_fixture_lab_context
from repo_rag_lab.notebook_support import (
    assert_contains_text,
    assert_minimum_pass_rate,
    assert_no_validation_issues,
    resolve_repo_root,
    write_notebook_run_log,
)

root = resolve_repo_root(Path.cwd().resolve())
context = build_hushwheel_fixture_lab_context(root)
context["fixture_root"]


## 2. Inspect Corpus Scale And Benchmark Inputs

### 2.1 Confirm The Fixture Is Large Enough To Stress Retrieval

Review the generated manifest, the corpus summary, and the question-suite summary before interpreting answers.

In [ ]:
{
    "manifest": context["fixture_manifest"],
    "corpus_summary": context["corpus_summary"],
    "training_summary": context["training_summary"],
    "population_summary": context["population_summary"],
}


## 3. Compare Concept And Code Retrieval

### 3.1 Review One Lore Question And One Implementation Question

The shared scaffold highlights a concept-oriented question and a code-oriented question so ranking shifts stay easy to interpret.

In [ ]:
[
    {
        "question": run["question"],
        "context_sources": run["context_sources"],
        "answer_preview": run["answer"][:280],
    }
    for run in context["highlight_runs"]
]


## 4. Assert Benchmark Health And Log The Run

### 4.1 Fail Fast On Retrieval Regressions

Keep the notebook useful as a playbook by asserting benchmark health, checking the highlighted answers, and writing a notebook log.

In [ ]:
assert_no_validation_issues(
    context["training_validation_issues"],
    context="hushwheel training samples",
)
assert_no_validation_issues(
    context["population_validation_issues"],
    context="hushwheel population samples",
)
assert_minimum_pass_rate(context["benchmark_summary"], minimum_pass_rate=1.0)
assert_contains_text(
    context["highlight_runs"][0]["answer"],
    ["ember index", "lantern vowel"],
    context=context["highlight_runs"][0]["question"],
)
assert_contains_text(
    context["highlight_runs"][1]["answer"],
    ["print_prefix_matches", "prefix search"],
    context=context["highlight_runs"][1]["question"],
)
log_path = write_notebook_run_log(root, "05-hushwheel-fixture-rag-lab", context)
{
    "benchmark_summary": context["benchmark_summary"],
    "reranked_sources": context["reranked_sources"],
    "log_path": str(log_path.relative_to(root)),
}
